In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader

# Configuración del dispositivo (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Versión de PyTorch: {torch.__version__}")
print(f"Dispositivo en uso: {device}")

Versión de PyTorch: 2.13.0+cpu
Dispositivo en uso: cpu


# Fundamentos y Dimensionalidad de Redes Recurrentes en PyTorch

En PyTorch, los tres bloques recurrentes principales implementados en `torch.nn` son:
- `nn.RNN`: Red Neuronal Recurrente simple (Vanilla).
- `nn.LSTM`: *Long Short-Term Memory* (con celda de memoria $c_t$ y compuertas de entrada, olvido y salida).
- `nn.GRU`: *Gated Recurrent Unit* (versión simplificada con compuertas de actualización y reinicio).

### Dimensiones clave de los Tensores (`batch_first=True`)

* **Entrada ($X$)**: `(batch_size, seq_len, input_size)`
* **Salida de la capa recurrente ($out$)**: `(batch_size, seq_len, hidden_size * num_directions)`
* **Estado Oculto ($h_n$)**: `(num_layers * num_directions, batch_size, hidden_size)`
* **Estado de Celda ($c_n$)** *(solo LSTM)*: `(num_layers * num_directions, batch_size, hidden_size)`

Veamos un ejemplo práctico con tensores dummy para verificar las dimensiones antes de pasar a modelos reales.

In [ ]:
# Parámetros de ejemplo
batch_size = 4
seq_len = 10
input_size = 8
hidden_size = 16
num_layers = 2

# Tensor de entrada sintético
x = torch.randn(batch_size, seq_len, input_size)
print(f'forma del tensor {x.shape}')

forma del tensor torch.Size([4, 10, 8])


In [ ]:
# 1. Prueba con nn.RNN
rnn = nn.RNN(input_size=input_size, hidden_size= hidden_size, num_layers = num_layers, batch_first = True)
outrnn, hnn_rnn = rnn(x)

In [ ]:
outrnn.shape

torch.Size([4, 10, 16])

In [ ]:
hnn_rnn.shape

torch.Size([2, 4, 16])

In [ ]:
# 2. Prueba con nn.LSTM
lstm = nn.LSTM(input_size=input_size, hidden_size= hidden_size, num_layers = num_layers, batch_first = True)
outlstm, (hnlstm, cnlstm) = lstm(x)

In [ ]:
outlstm.shape

torch.Size([4, 10, 16])

In [ ]:
hnlstm.shape

torch.Size([2, 4, 16])

In [ ]:
cnlstm.shape

torch.Size([2, 4, 16])

In [ ]:
# 3. Prueba con nn.GRU
gru = nn.GRU(input_size=input_size, hidden_size= hidden_size, num_layers = num_layers, batch_first = True)
outgru, hnngru = gru(x)


In [ ]:
outgru.shape

torch.Size([4, 10, 16])

In [ ]:
hnngru.shape

torch.Size([2, 4, 16])


# Ejemplo 1: Predicción de Series Temporales con LSTM (Google Stock Price)

En este ejemplo construiremos un modelo de regresión recurrente utilizando **PyTorch** para predecir el precio de apertura (*Open*) de las acciones de Google basándonos en una ventana temporal de los 60 días anteriores ($t-60$ a $t-1$).

### Flujo de trabajo en PyTorch:
1. **Carga y preprocesamiento**: Escalamiento de datos con `MinMaxScaler`.
2. **Construcción del Dataset**: Clase `Dataset` personalizada de PyTorch y `DataLoader`.
3. **Definición de la arquitectura**: `nn.Module` con capas apiladas de `nn.LSTM` y `nn.Dropout`.
4. **Ciclo de entrenamiento**: Backpropagation con `MSELoss` y `Adam`.
5. **Evaluación y pronóstico**: Inversión de escala y comparación gráfica con `matplotlib`.

In [ ]:
import os

# Carga del conjunto de entrenamiento
train_path = 'Data/Google_Stock_Price_Train.csv'
dataset_train = pd.read_csv(train_path)

print(f"Dimensiones del dataset de entrenamiento: {dataset_train.shape}")
display(dataset_train.head())

# Extraemos la columna 'Open'
training_data = dataset_train[['Open']].values

# Normalización de los datos entre 0 y 1
scaler = MinMaxScaler(feature_range=(0, 1))
training_data_scaled = scaler.fit_transform(training_data)

Dimensiones del dataset de entrenamiento: (1258, 6)


,Date,Open,High,Low,Close,Volume
0,1/3/2012,325.25,332.83,324.97,663.59,"7,380,500"
1,1/4/2012,331.27,333.87,329.08,666.45,"5,749,400"
2,1/5/2012,329.83,330.75,326.89,657.21,"6,590,300"
3,1/6/2012,328.34,328.77,323.68,648.24,"5,405,900"
4,1/9/2012,322.04,322.29,309.46,620.76,"11,688,800"


In [ ]:
# Crear las secuencias temporales de longitud 60
SEQ_LENGTH = 60

def create_sliding_windows(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i,0])

    return np.array(X), np.array(y)

#Train test data
X_train_np, y_train_np = create_sliding_windows(training_data_scaled, SEQ_LENGTH)

In [ ]:
pd.DataFrame(X_train_np)

,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.085814,0.097012,0.094334,0.091562,0.079842,0.064328,0.058542,0.065686,0.061091,0.066393,...,0.052143,0.056124,0.058189,0.065407,0.068830,0.072438,0.079935,0.078466,0.080345,0.084977
1,0.097012,0.094334,0.091562,0.079842,0.064328,0.058542,0.065686,0.061091,0.066393,0.061426,...,0.056124,0.058189,0.065407,0.068830,0.072438,0.079935,0.078466,0.080345,0.084977,0.086279
2,0.094334,0.091562,0.079842,0.064328,0.058542,0.065686,0.061091,0.066393,0.061426,0.074745,...,0.058189,0.065407,0.068830,0.072438,0.079935,0.078466,0.080345,0.084977,0.086279,0.084716
3,0.091562,0.079842,0.064328,0.058542,0.065686,0.061091,0.066393,0.061426,0.074745,0.027978,...,0.065407,0.068830,0.072438,0.079935,0.078466,0.080345,0.084977,0.086279,0.084716,0.074541
4,0.079842,0.064328,0.058542,0.065686,0.061091,0.066393,0.061426,0.074745,0.027978,0.023793,...,0.068830,0.072438,0.079935,0.078466,0.080345,0.084977,0.086279,0.084716,0.074541,0.078838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1193,0.934445,0.924939,0.921069,0.924381,0.930482,0.929905,0.931133,0.927506,0.944155,0.938760,...,0.931766,0.941141,0.957623,0.964134,0.964023,0.969715,0.950778,0.962944,0.961232,0.954759
1194,0.924939,0.921069,0.924381,0.930482,0.929905,0.931133,0.927506,0.944155,0.938760,0.934035,...,0.941141,0.957623,0.964134,0.964023,0.969715,0.950778,0.962944,0.961232,0.954759,0.952043
1195,0.921069,0.924381,0.930482,0.929905,0.931133,0.927506,0.944155,0.938760,0.934035,0.934835,...,0.957623,0.964134,0.964023,0.969715,0.950778,0.962944,0.961232,0.954759,0.952043,0.951633
1196,0.924381,0.930482,0.929905,0.931133,0.927506,0.944155,0.938760,0.934035,0.934835,0.931394,...,0.964134,0.964023,0.969715,0.950778,0.962944,0.961232,0.954759,0.952043,0.951633,0.957251


In [ ]:
# Dataset personalizado en PyTorch
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # PyTorch espera tensores flotantes: (batch_size, seq_len, input_size)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
#train_dataset, and dataloader
train_dataset = TimeSeriesDataset(X_train_np, y_train_np)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)


In [ ]:
train_dataset.X.shape

torch.Size([1198, 60, 1])

In [ ]:
train_dataset.y.shape

torch.Size([1198, 1])

### Definición de la Red Neuronal Recurrente (LSTM) en PyTorch

En PyTorch heredamos de `nn.Module` e implementamos el método `forward`.
Utilizaremos:
- Una capa `nn.LSTM` con `num_layers=3`, `hidden_size=50` y `dropout=0.2`.
- Una capa completamente conectada `nn.Linear(hidden_size, 1)` para obtener la predicción escalar del precio.

In [ ]:
class LSTMStockRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=3, dropout=0.2):
        super(LSTMStockRegressor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Capa recurrente LSTM apilada
        self.lstm = nn.LSTM(input_size=input_size, hidden_size= hidden_size,
                            num_layers = num_layers, batch_first = True, dropout = dropout if num_layers > 1 else 0.0)

        # Capa densa de salida
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)

        # Tomamos el estado oculto del último instante de tiempo (out[:, -1, :])
        last_time_step = out[:,-1,:]

        # Predicción final
        prediction = self.fc(last_time_step)
        return prediction

# Instanciar el modelo y mover al dispositivo (CPU/GPU)
model_lstm = LSTMStockRegressor().to(device)
print(model_lstm)

LSTMStockRegressor(
  (lstm): LSTM(1, 50, num_layers=3, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)


### Entrenamiento del Modelo con PyTorch

Definimos:
- Función de pérdida: `nn.MSELoss()` (Error Cuadrático Medio).
- Optimizador: `torch.optim.Adam` con tasa de aprendizaje `lr=0.001`.
- Entrenamiento por 50 épocas con registro del historial de pérdida.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

epochs = 50
train_losses = []

model_lstm.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # 1. Limpiar gradientes
        optimizer.zero_grad()

        # 2. Paso hacia adelante (Forward pass)
        outputs = model_lstm(batch_X)
        loss = criterion(outputs, batch_y)

        # 3. Retropropagación (Backward pass)
        loss.backward()

        # 4. Actualización de pesos
        optimizer.step()

        epoch_loss += loss.item()*batch_X.size(0)

    #Epocas, losses
    epoch_loss /= len(train_loader.dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1)%10 ==0 or epoch == 0:
        print(f'Epoca[{epoch + 1}/{epochs}] --- MSE Loss: {epoch_loss}')

Epoca[1/50] --- MSE Loss: 0.14217791173961605
Epoca[10/50] --- MSE Loss: 0.001838466944717157
Epoca[20/50] --- MSE Loss: 0.0016650307225063443
Epoca[30/50] --- MSE Loss: 0.0012177710395340554
Epoca[40/50] --- MSE Loss: 0.0011573773237389944
Epoca[50/50] --- MSE Loss: 0.000989729709393082


In [ ]:
# Gráfica de la función de pérdida
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Pérdida de Entrenamiento (MSE)', color='blue')
plt.title('Evolución de la Pérdida durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.grid(True)
plt.legend()
plt.show()

### Evaluación y Pronóstico sobre el Conjunto de Prueba

Para evaluar:
1. Cargamos el conjunto de prueba (`Google_Stock_Price_Test.csv`).
2. Concatenamos el histórico necesario (los 60 días anteriores del train set) para alimentar las ventanas del test set.
3. Transformamos a tensores de PyTorch y ejecutamos `model_lstm.eval()` con `torch.no_grad()`.
4. Invertimos el escalamiento con `scaler.inverse_transform` y graficamos la comparación.

In [ ]:
# 1. Cargar datos de prueba
test_path = 'Data/Google_Stock_Price_Test.csv'
dataset_test = pd.read_csv(test_path)
real_stock_price = dataset_test[['Open']].values

# 2. Concatenar para tener el histórico de 60 timesteps antes de la primera fecha de test
dataset_total = pd.concat((dataset_train['Open'], dataset_test['Open']), axis=0)
inputs = dataset_total[len(dataset_total) - len(dataset_test) - SEQ_LENGTH:].values
inputs = inputs.reshape(-1, 1)
inputs_scaled = scaler.transform(inputs)


In [ ]:
SEQ_LENGTH = 60

def create_sliding_windows(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i,0])

    return np.array(X), np.array(y)

#Train test data


In [ ]:
# 3. Construir ventanas para el test set
X_test_np, y_test_np = create_sliding_windows(inputs_scaled, SEQ_LENGTH)

In [ ]:
X_test_np.shape

(20, 60)

In [ ]:
# Tensor PyTorch (batch_size, seq_len, input_size)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32).unsqueeze(-1).to(device)
X_test_tensor.shape

torch.Size([20, 60, 1])

In [ ]:
# 4. Inferencia con PyTorch
model_lstm.eval()
with torch.no_grad():
    predicted_stock_price_tensor = model_lstm(X_test_tensor)
    predicted_stock_price_scaled = predicted_stock_price_tensor.cpu().numpy()



In [ ]:
# 5. Invertir escalamiento
predicted_stock_price = scaler.inverse_transform(predicted_stock_price_scaled)
predicted_stock_price

array([[780.6108 ],
       [778.9066 ],
       [777.65564],
       [776.78687],
       [776.8239 ],
       [778.22754],
       [780.58   ],
       [783.1309 ],
       [785.6201 ],
       [787.8375 ],
       [789.6455 ],
       [790.94904],
       [791.76917],
       [792.324  ],
       [792.7173 ],
       [793.903  ],
       [796.0826 ],
       [799.2645 ],
       [802.6552 ],
       [804.6384 ]], dtype=float32)

In [ ]:
# 6. Visualización
plt.figure(figsize=(10, 5))
plt.plot(real_stock_price, color='red', label='Precio Real de las Acciones de Google')
plt.plot(predicted_stock_price, color='blue', linestyle='--', label='Predicción con PyTorch LSTM')
plt.title('Predicción de Precios de Acciones de Google con PyTorch')
plt.xlabel('Tiempo (Días de Prueba)')
plt.ylabel('Precio de Apertura (USD)')
plt.legend()
plt.grid(True)
plt.show()


# Ejemplo 2: Clasificación de Sentimientos con GRU Bidireccional y Embeddings

En tareas de Procesamiento de Lenguaje Natural (PLN), las redes recurrentes procesan secuencias de palabras tokenizadas.

### Elementos clave:
1. **Capa de Embeddings (`nn.Embedding`)**: Mapea índices enteros de palabras a vectores densos continuos en $\mathbb{R}^{d}$.
2. **GRU Bidireccional (`bidirectional=True`)**: Procesa el texto en dos direcciones:
   - **Hacia adelante ($\rightarrow$)**: Captura el contexto pasado.
   - **Hacia atrás ($\leftarrow$)**: Captura el contexto futuro.
3. **Concatenación de Estados**: Se combinan los estados ocultos finales de ambas direcciones para la decisión de clasificación.

In [ ]:
import re
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader

# Dataset de ejemplo para análisis de sentimientos (positivo=1, negativo=0)
corpus = [
    ("este modelo es increible y funciona de maravilla", 1),
    ("excelente rendimiento y resultados muy precisos", 1),
    ("me encanta la precision de esta red neuronal", 1),
    ("una arquitectura fantastica muy facil de entrenar", 1),
    ("gran implementacion super util para mi proyecto", 1),
    ("los resultados superaron todas mis expectativas", 1),
    ("muy buena explicacion y codigo perfectamente claro", 1),
    ("servicio pesimo el modelo no aprende nada", 0),
    ("terrible precision y muchos errores en prediccion", 0),
    ("un desastre total no converge y pierde tiempo", 0),
    ("muy mal rendimiento pesima eleccion de hiperparametros", 0),
    ("error constante y gradientes que explotan", 0),
    ("horrible experiencia muy dificil de configurar", 0),
    ("la peor red neuronal que he probado en mi vida", 0)
]


In [ ]:
# 1. Tokenizador básico
def tokenize(text):
    text = text.lower()
    return re.findall(r'\b\w+\b', text)


In [ ]:
# 2. Construir Vocabulario
tokens_list = [tokenize(text) for text, _ in corpus]
counter = Counter([token for tokens in tokens_list for token in tokens])

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in counter.items():
    vocab[word] = len(vocab)

print(f"Tamaño del vocabulario: {len(vocab)} palabras")


Tamaño del vocabulario: 78 palabras


In [ ]:
# 3. Vectorización y Padding
MAX_SEQ_LEN = 10

def encode_and_pad(text, vocab, max_len=MAX_SEQ_LEN):
    tokens = tokenize(text)
    indices = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    if len(indices) < max_len:
        indices += [vocab["<PAD>"]] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

X_nlp = np.array([encode_and_pad(text, vocab) for text, _ in corpus])
y_nlp = np.array([label for _, label in corpus])

# Dataset y DataLoader
nlp_dataset = TensorDataset(torch.tensor(X_nlp, dtype=torch.long), torch.tensor(y_nlp, dtype=torch.float32).unsqueeze(1))
nlp_loader = DataLoader(nlp_dataset, batch_size=4, shuffle=True)

print(f"Forma de entrada X: {X_nlp.shape}")
print(f"Forma de etiquetas y: {y_nlp.shape}")

Forma de entrada X: (14, 10)
Forma de etiquetas y: (14,)


### Arquitectura del Clasificador con GRU Bidireccional

Implementamos `BiGRUSentimentClassifier`:
- Capa `Embedding` con índice de relleno (`padding_idx=0`).
- Capa `GRU` bidireccional de 2 capas.
- Concatenación del último estado oculto hacia adelante ($h_{\rightarrow}$) y hacia atrás ($h_{\leftarrow}$).
- Capa totalmente conectada lineal con `Dropout`.

In [ ]:
class BiGRUSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=2, dropout=0.3):
        super(BiGRUSentimentClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        # Multiplicamos por 2 debido a que es bidireccional
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, text_indices):
        # text_indices: (batch_size, seq_len)
        embedded = self.embedding(text_indices)  # (batch_size, seq_len, embed_dim)

        out, h_n = self.gru(embedded)
        # h_n tiene forma: (num_layers * 2, batch_size, hidden_size)

        # Obtenemos el último estado forward y backward de la capa superior
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]

        # Concatenamos ambas direcciones
        h_combined = torch.cat((h_forward, h_backward), dim=1)  # (batch_size, hidden_size * 2)

        logits = self.fc(self.dropout(h_combined))
        return logits

# Instanciar modelo
model_nlp = BiGRUSentimentClassifier(vocab_size=len(vocab)).to(device)
print(model_nlp)

BiGRUSentimentClassifier(
  (embedding): Embedding(78, 32, padding_idx=0)
  (gru): GRU(32, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
criterion_nlp = nn.BCEWithLogitsLoss()
optimizer_nlp = torch.optim.Adam(model_nlp.parameters(), lr=0.01)

epochs_nlp = 60
model_nlp.train()

for epoch in range(epochs_nlp):
    epoch_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in nlp_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer_nlp.zero_grad()
        logits = model_nlp(batch_x)
        loss = criterion_nlp(logits, batch_y)
        loss.backward()
        optimizer_nlp.step()

        epoch_loss += loss.item() * batch_x.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    epoch_loss /= total
    accuracy = correct / total

    if (epoch + 1) % 15 == 0 or epoch == 0:
        print(f"Época [{epoch+1}/{epochs_nlp}] - Pérdida: {epoch_loss:.4f} | Precisión: {accuracy * 100:.2f}%")

Época [1/60] - Pérdida: 0.8154 | Precisión: 28.57%
Época [15/60] - Pérdida: 0.0000 | Precisión: 100.00%
Época [30/60] - Pérdida: 0.0000 | Precisión: 100.00%
Época [45/60] - Pérdida: 0.0000 | Precisión: 100.00%
Época [60/60] - Pérdida: 0.0000 | Precisión: 100.00%


In [ ]:
# Función de inferencia para nuevas frases
def predict_sentiment(text, model, vocab, max_len=MAX_SEQ_LEN):
    model.eval()
    encoded = encode_and_pad(text, vocab, max_len)
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.sigmoid(logits).item()

    sentiment = "Positivo" if prob >= 0.5 else "Negativo"
    print(f"Texto: '{text}'")
    print(f"Probabilidad Positiva: {prob:.4f} -> Predicción: {sentiment}\n")

# Pruebas con frases nuevas
predict_sentiment("este algoritmo es fantastico y rapido", model_nlp, vocab)
predict_sentiment("terrible error muy mal servicio", model_nlp, vocab)
predict_sentiment("excelente implementacion de red recurrente", model_nlp, vocab)

Texto: 'este algoritmo es fantastico y rapido'
Probabilidad Positiva: 0.7121 -> Predicción: Positivo

Texto: 'terrible error muy mal servicio'
Probabilidad Positiva: 0.0000 -> Predicción: Negativo

Texto: 'excelente implementacion de red recurrente'
Probabilidad Positiva: 0.0005 -> Predicción: Negativo



# Ejemplo 3: Generación de Texto a Nivel de Caracteres con LSTM

Un modelo de lenguaje autorregresivo a nivel de caracteres aprende la distribución condicional de probabilidad:
$$P(c_{t+1} \mid c_1, c_2, \dots, c_t)$$

Dado un texto de entrada, el modelo aprende la estructura sintáctica y semántica para generar nuevas secuencias paso a paso utilizando **muestreo con temperatura**:
$$P_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$
Donde $T$ (temperatura) controla la creatividad vs determinismo del texto generado.

In [ ]:
# Texto de muestra para el entrenamiento del modelo de lenguaje
text_data = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivia un hidalgo de los de lanza en astillero, adarga antigua, rocin flaco y galgo corredor. Una olla de algo mas vaca que carnero, salpicon las mas noches, duelos y quebrantos los sabados, lantejas los viernes, algun palomino de anadidura los domingos, consumian las tres partes de su hacienda. El resto della concluian sayo de velarte, calzas de velludo para las fiestas, con sus pantuflos de lo mesmo, y los dias de entresemana se honraba con su vellori de lo mas fino. Tenia en su casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un mozo de campo y plaza, que asi ensillaba el rocin como tomaba la podadera. Frisaba la edad de nuestro hidalgo con los cincuenta anos; era de complexion recia, seco de carnes, enjuto de rostro, gran madrugador y amigo de la caza."""

# Crear vocabulario a nivel de caracteres
chars = sorted(list(set(text_data)))
vocab_size_gen = len(chars)
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}

print(f"Total de caracteres en el texto: {len(text_data)}")
print(f"Total de caracteres únicos (vocabulario): {vocab_size_gen}")

# Construir secuencias de longitud fija (input y target desplazado 1 carácter)
CHUNK_LEN = 40

def create_char_sequences(text, chunk_len):
    inputs, targets = [], []
    for i in range(0, len(text) - chunk_len, 3):
        in_chunk = [char2idx[ch] for ch in text[i : i + chunk_len]]
        tar_chunk = [char2idx[ch] for ch in text[i + 1 : i + chunk_len + 1]]
        inputs.append(in_chunk)
        targets.append(tar_chunk)
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

inputs_gen, targets_gen = create_char_sequences(text_data, CHUNK_LEN)
gen_dataset = TensorDataset(inputs_gen, targets_gen)
gen_loader = DataLoader(gen_dataset, batch_size=16, shuffle=True)

print(f"Número de secuencias de entrenamiento: {len(inputs_gen)}")

Total de caracteres en el texto: 887
Total de caracteres únicos (vocabulario): 33
Número de secuencias de entrenamiento: 283


In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=128, num_layers=2, dropout=0.2):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)
        out, hidden = self.lstm(embedded, hidden)
        logits = self.fc(out)  # (batch_size, seq_len, vocab_size)
        return logits, hidden

    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)

model_gen = CharLSTM(vocab_size=vocab_size_gen).to(device)
print(model_gen)

CharLSTM(
  (embedding): Embedding(33, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=128, out_features=33, bias=True)
)


In [ ]:
criterion_gen = nn.CrossEntropyLoss()
optimizer_gen = torch.optim.Adam(model_gen.parameters(), lr=0.005)

epochs_gen = 100
model_gen.train()

for epoch in range(epochs_gen):
    total_loss = 0.0
    for batch_x, batch_y in gen_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer_gen.zero_grad()
        logits, _ = model_gen(batch_x)

        # Redimensionar para CrossEntropyLoss: (batch * seq_len, vocab_size) y (batch * seq_len)
        loss = criterion_gen(logits.view(-1, vocab_size_gen), batch_y.view(-1))
        loss.backward()

        # Gradient Clipping para evitar explosión de gradientes
        nn.utils.clip_grad_norm_(model_gen.parameters(), max_norm=5.0)
        optimizer_gen.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0 or epoch == 0:
        avg_loss = total_loss / len(gen_loader)
        print(f"Época [{epoch+1}/{epochs_gen}] - Pérdida de Lenguaje: {avg_loss:.4f}")

Época [1/100] - Pérdida de Lenguaje: 2.8696
Época [20/100] - Pérdida de Lenguaje: 0.1205
Época [40/100] - Pérdida de Lenguaje: 0.0894
Época [60/100] - Pérdida de Lenguaje: 0.0849
Época [80/100] - Pérdida de Lenguaje: 0.0812
Época [100/100] - Pérdida de Lenguaje: 0.0869


In [ ]:
def generate_text(model, start_str="En un lugar", length=300, temperature=0.7):
    model.eval()
    chars_generated = list(start_str)

    # Vectorizar el texto inicial
    input_indices = torch.tensor([[char2idx.get(ch, 0) for ch in start_str]], dtype=torch.long).to(device)
    hidden = None

    with torch.no_grad():
        # Pasar el texto inicial para calentar el estado oculto
        out, hidden = model(input_indices, hidden)

        # Primer caracter predicho
        last_logit = out[:, -1, :] / temperature
        probs = torch.softmax(last_logit, dim=-1)
        next_char_idx = torch.multinomial(probs, num_samples=1).item()
        chars_generated.append(idx2char[next_char_idx])

        # Bucle de generación autorregresiva
        current_input = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
        for _ in range(length):
            out, hidden = model(current_input, hidden)
            probs = torch.softmax(out[:, -1, :] / temperature, dim=-1)
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            chars_generated.append(idx2char[next_char_idx])
            current_input = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)

    return "".join(chars_generated)

# Generación con diferentes temperaturas
for temp in [0.3, 0.7, 1.2]:
    print(f"\n{'='*20} TEMPERATURA {temp} {'='*20}")
    print(generate_text(model_gen, start_str="En un lugar de la", length=250, temperature=temp))


==================== TEMPERATURA 0.3 ====================
En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivia un hidalgo de los de lanza en astillero, adarga antigua, rocin flaco y galgo corredor. Una olla de algo mas vaca que carnero, salpicon las mas noches, duelos y quebrantos los sa

==================== TEMPERATURA 0.7 ====================
En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivia un hidalgo de los de lanza en astillero, adarga antigua, rocin flaco y galgo corredor. Una olla de algo mas vaca que carnero, salpicon las mas noches, duelos y quebrantos los sa

==================== TEMPERATURA 1.2 ====================
En un lugar de la Mancha, be velludo para las fiestas, con sus pantuflos de lo mesmo, y los dias de entresemana se honraba con su vellori de lo mas fino. Tenia en su casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un mozo de campo
